# Transformed-response conformal intervals

Distributional conformal intervals (prediction intervals calibrated on held-out rows from a fitted predictive distribution) use the units of the observed response, including when the family is fitted to `log(y)`, `center(y)` or `scale(y)`. This diagnostic compares both calibration routes with independently computed Gaussian or lognormal quantiles on exactly matched partitions. The corrected intervals agree within 9 × 10⁻¹⁶ response units in all 12 cases; without the correction, the internal route (which refits the formula on proper-training rows) has errors between 1.48 and 2.04 units, and the external route (which calibrates a model the caller has already fitted) raises an exception for `center(y)`.

The headline table reports every transform, calibration route and stopping prefix (the fit read at a given boosting step). The fixed [protocol](lss_response_scale/protocol.md), [baseline record](lss_response_scale/baseline.json) and [confirmation record](lss_response_scale/corrected.json) retain row identities, endpoint arrays and source hashes. This is a deterministic correctness diagnostic, with no sampling uncertainty estimate or competitor ranking.

In [1]:
from pathlib import Path
import json
import sys
import warnings

import numpy as np
import polars as pl
from great_tables import GT, md

root = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "Cargo.toml").exists())
sys.path.insert(0, str(root / "python"))
sys.path.insert(0, str(root))
import mboostrs as mb
from benches.audit_lss_response_scale import inputs, run, subset

directory = root / "docs/book/benchmarks/lss_response_scale"
before = json.loads((directory / "baseline.json").read_text())
after = json.loads((directory / "corrected.json").read_text())

In [2]:
assert before["data_sha256"] == after["data_sha256"]
assert before["training_rows"] == after["training_rows"]
assert before["calibration_rows"] == after["calibration_rows"]
rows = []
for old, new in zip(before["records"], after["records"], strict=True):
    keys = ("transform", "mstop", "route")
    assert all(old[key] == new[key] for key in keys)
    rows.append({"Response": old["transform"] + "(y)", "Steps": old["mstop"] or 60,
                 "Calibration": old["route"],
                 "Earlier": "exception" if "error" in old else f'{old["max_absolute_error"]:.3e}',
                 "Corrected": f'**{new["max_absolute_error"]:.3e}**'})
comparison = pl.DataFrame(rows)

In [3]:
(GT(comparison)
 .tab_header(title="Original-response interval accuracy",
             subtitle="Maximum absolute endpoint/centre error in response units; smaller is better")
 .tab_spanner(label="Error against analytical quantiles", columns=["Earlier", "Corrected"])
 .fmt_markdown(columns="Corrected")
 .tab_source_note(md("Each row uses the same three query values and 150/150 fit/calibration partition. "
                     "Bold denotes the corrected result, accurate within 1e-10. Earlier source: 0d94682. "
                     "The external center(y) exception is tracked separately in issue #639. "
                     "These are numerical identities on one fixed dataset, with no Monte Carlo SE.")))

GT(_tbl_data=shape: (12, 5)
┌───────────┬───────┬─────────────┬───────────┬───────────────┐
│ Response  ┆ Steps ┆ Calibration ┆ Earlier   ┆ Corrected     │
│ ---       ┆ ---   ┆ ---         ┆ ---       ┆ ---           │
│ str       ┆ i64   ┆ str         ┆ str       ┆ str           │
╞═══════════╪═══════╪═════════════╪═══════════╪═══════════════╡
│ log(y)    ┆ 60    ┆ internal    ┆ 1.484e+00 ┆ **2.220e-16** │
│ log(y)    ┆ 60    ┆ external    ┆ 2.220e-16 ┆ **2.220e-16** │
│ log(y)    ┆ 30    ┆ internal    ┆ 1.507e+00 ┆ **4.441e-16** │
│ log(y)    ┆ 30    ┆ external    ┆ 4.441e-16 ┆ **4.441e-16** │
│ center(y) ┆ 60    ┆ internal    ┆ 1.482e+00 ┆ **4.441e-16** │
│ …         ┆ …     ┆ …           ┆ …         ┆ …             │
│ center(y) ┆ 30    ┆ external    ┆ exception ┆ **2.220e-16** │
│ scale(y)  ┆ 60    ┆ internal    ┆ 2.034e+00 ┆ **8.882e-16** │
│ scale(y)  ┆ 60    ┆ external    ┆ 8.882e-16 ┆ **8.882e-16** │
│ scale(y)  ┆ 30    ┆ internal    ┆ 2.037e+00 ┆ **2.220e-16** │
│ scale(y)  ┆ 30    ┆ external    ┆ 2.220e-16 ┆ **2.220e-16** │
└───────────┴───────┴─────────────┴───────────┴───────────────┘, _body=<great_tables._gt_data.Body object at 0x1140369f0>, _boxhead=Boxhead([ColInfo(var='Response', type=<ColInfoTypeEnum.default: 1>, column_label='Response', column_align='left', column_width=None), ColInfo(var='Steps', type=<ColInfoTypeEnum.default: 1>, column_label='Steps', column_align='right', column_width=None), ColInfo(var='Calibration', type=<ColInfoTypeEnum.default: 1>, column_label='Calibration', column_align='left', column_width=None), ColInfo(var='Earlier', type=<ColInfoTypeEnum.default: 1>, column_label='Earlier', column_align='left', column_width=None), ColInfo(var='Corrected', type=<ColInfoTypeEnum.default: 1>, column_label='Corrected', column_align='left', column_width=None)]), _stub=<great_tables._gt_data.Stub object at 0x114035ee0>, _spanners=Spanners([SpannerInfo(spanner_id='Error against analytical quantiles', spanner_level=0, spanner_label='Error against analytical quantiles', spanner_units=None, spanner_pattern=None, vars=['Earlier', 'Corrected'], built=None)]), _heading=Heading(title='Original-response interval accuracy', subtitle='Maximum absolute endpoint/centre error in response units; smaller is better', preheader=None), _stubhead=None, _summary_rows=<great_tables._gt_data.SummaryRows object at 0x112ebcd70>, _summary_rows_grand=<great_tables._gt_data.SummaryRows object at 0x10e5987d0>, _source_notes=[Md(text='Each row uses the same three query values and 150/150 fit/calibration partition. Bold denotes the corrected result, accurate within 1e-10. Earlier source: 0d94682. The external center(y) exception is tracked separately in issue #639. These are numerical identities on one fixed dataset, with no Monte Carlo SE.')], _footnotes=[], _styles=[], _locale=<great_tables._gt_data.Locale object at 0x113fc8140>, _formats=[<great_tables._gt_data.FormatInfo object at 0x1140361b0>], _substitutions=[], _col_merge=[], _transforms=[], _options=Options(table_id=OptionsInfo(scss=False, category='table', type='value', value=None), table_caption=OptionsInfo(scss=False, category='table', type='value', value=None), table_width=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_layout=OptionsInfo(scss=True, category='table', type='value', value='fixed'), table_margin_left=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_margin_right=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_background_color=OptionsInfo(scss=True, category='table', type='value', value='#FFFFFF'), table_additional_css=OptionsInfo(scss=False, category='table', type='values', value=[]), table_font_names=OptionsInfo(scss=False, category='table', type='values', value=['-apple-system', 'BlinkMacSystemFont', 'Segoe UI', 'Roboto', 'Oxygen', 'Ubuntu', 'Cantarell', 'Helvetica Neue', 'Fira Sans', 'Droid Sans', 'Arial', 'sans-serif']), table_font_size=OptionsInfo(scss=True, category='table', ty

## Data and partition

The observed response is positive and dimensionless: $Y=\exp(0.3+0.6X+0.15Z)$, with independent $X\sim U(-1,1)$ and $Z\sim N(0,1)$. Seed 6319 gives 300 rows. Seed 9 permutes those rows, assigning the first 150 to calibration and the remainder to proper training. Every formula and the 60-iteration budget are fixed in advance. The 30-iteration prefix checks that calibration reads the requested path position.

A Gaussian family fitted to `log(y)` implies a lognormal distribution for the original response. Its quantiles are $\exp(\mu+\sigma\Phi^{-1}(p))$. For affine response transforms, the inverse uses the proper-training response mean and, for `scale`, its sample standard deviation. The independent reference applies these inverses **before** computing original-response calibration scores.

In [4]:
data, query, training_rows, calibration_rows = inputs()
training = subset(data, training_rows)
calibration = subset(data, calibration_rows)
formula = {"mu": "log(y) ~ bols(x)", "sigma": "log(y) ~ bols(x)"}

model = mb.gamboostLSS(formula, data, family=mb.GaussianLSS(),
                      control=mb.boost_control(60))
interval = mb.conformal_interval(model, query, level=.9, seed=9)
assert interval.fitted_mean is None
assert interval.n_training == interval.n_calibration == 150

## Internal and external calibration

The internal route refits the raw formula on proper-training rows. This re-estimates response centering/scaling and all fitted predictor preprocessing there. It then inverse-transforms the refit's quantiles and scores the original calibration responses. An explicit `calibration=` frame instead calibrates the model already fitted by the caller, with no further refit or split. The two routes agree when given the same specification, partition and path prefix.

In [5]:
proper = mb.gamboostLSS(formula, training, family=mb.GaussianLSS(),
                       control=mb.boost_control(60))
external = mb.conformal_interval(proper, query, calibration=calibration, level=.9)
np.testing.assert_allclose(interval.lower, external.lower, atol=1e-10, rtol=0)
np.testing.assert_allclose(interval.upper, external.upper, atol=1e-10, rtol=0)

reported = pl.DataFrame({"x": query["x"], "Lower": interval.lower,
                         "Centre": interval.predicted, "Upper": interval.upper})
(GT(reported)
 .tab_header(title="Calibrated intervals for the original response",
             subtitle="90% marginal prediction intervals; endpoints and centre in response units")
 .fmt_number(columns=["Lower", "Centre", "Upper"], decimals=6)
 .tab_source_note("The centre is the proper-training median clipped into the calibrated band. "
                  "It is not the original-response mean."))

GT(_tbl_data=shape: (3, 4)
┌──────┬──────────┬──────────┬──────────┐
│ x    ┆ Lower    ┆ Centre   ┆ Upper    │
│ ---  ┆ ---      ┆ ---      ┆ ---      │
│ f64  ┆ f64      ┆ f64      ┆ f64      │
╞══════╪══════════╪══════════╪══════════╡
│ -0.5 ┆ 0.783242 ┆ 1.159736 ┆ 1.694945 │
│ 0.0  ┆ 0.923242 ┆ 1.361554 ┆ 1.985875 │
│ 0.5  ┆ 1.087656 ┆ 1.598491 ┆ 2.327325 │
└──────┴──────────┴──────────┴──────────┘, _body=<great_tables._gt_data.Body object at 0x113f685f0>, _boxhead=Boxhead([ColInfo(var='x', type=<ColInfoTypeEnum.default: 1>, column_label='x', column_align='right', column_width=None), ColInfo(var='Lower', type=<ColInfoTypeEnum.default: 1>, column_label='Lower', column_align='right', column_width=None), ColInfo(var='Centre', type=<ColInfoTypeEnum.default: 1>, column_label='Centre', column_align='right', column_width=None), ColInfo(var='Upper', type=<ColInfoTypeEnum.default: 1>, column_label='Upper', column_align='right', column_width=None)]), _stub=<great_tables._gt_data.Stub object at 0x1140369c0>, _spanners=Spanners([]), _heading=Heading(title='Calibrated intervals for the original response', subtitle='90% marginal prediction intervals; endpoints and centre in response units', preheader=None), _stubhead=None, _summary_rows=<great_tables._gt_data.SummaryRows object at 0x10e76b140>, _summary_rows_grand=<great_tables._gt_data.SummaryRows object at 0x10e76aae0>, _source_notes=['The centre is the proper-training median clipped into the calibrated band. It is not the original-response mean.'], _footnotes=[], _styles=[], _locale=<great_tables._gt_data.Locale object at 0x113fc8740>, _formats=[<great_tables._gt_data.FormatInfo object at 0x114272510>], _substitutions=[], _col_merge=[], _transforms=[], _options=Options(table_id=OptionsInfo(scss=False, category='table', type='value', value=None), table_caption=OptionsInfo(scss=False, category='table', type='value', value=None), table_width=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_layout=OptionsInfo(scss=True, category='table', type='value', value='fixed'), table_margin_left=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_margin_right=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_background_color=OptionsInfo(scss=True, category='table', type='value', value='#FFFFFF'), table_additional_css=OptionsInfo(scss=False, category='table', type='values', value=[]), table_font_names=OptionsInfo(scss=False, category='table', type='values', value=['-apple-system', 'BlinkMacSystemFont', 'Segoe UI', 'Roboto', 'Oxygen', 'Ubuntu', 'Cantarell', 'Helvetica Neue', 'Fira Sans', 'Droid Sans', 'Arial', 'sans-serif']), table_font_size=OptionsInfo(scss=True, category='table', type='px', value='16px'), table_font_weight=OptionsInfo(scss=True, category='table', type='value', value='normal'), table_font_style=OptionsInfo(scss=True, category='table', type='value', value='normal'), table_font_color=OptionsInfo(scss=True, category='table', type='value', value='#333333'), table_font_color_light=OptionsInfo(scss=True, category='table', type='value', value='#FFFFFF'), table_border_top_include=OptionsInfo(scss=False, category='table', type='boolean', value=True), table_border_top_style=OptionsInfo(scss=True, category='table', type='value', value='solid'), table_border_top_width=OptionsInfo(scss=True, category='table', type='px', value='2px'), table_border_top_color=OptionsInfo(scss=True, category='table', type='value', value='#A8A8A8'), table_border_right_style=OptionsInfo(scss=True, category='table', type='value', value='none'), table_border_right_width=OptionsInfo(scss=True, category='table', type='px', value='2px'), table_border_right_color=OptionsInfo(scss=True, category='table', type='value', value='#D3D3D3'), table_border_bottom_include=OptionsInfo(scss=False, category='table', type='boolean', value=True), table_border_bottom_style=OptionsInfo(scss=True, category='table', type='value', value='solid'), table_border_bottom_width=Optio

## Confirmation and limits

The confirmation repeats all 12 cases and compares them with the confirmation record [corrected.json](lss_response_scale/corrected.json). Regression tests additionally change only calibration responses and verify unchanged proper-training response transforms, predictor recipes, coefficients and predictions. They exercise imputation, absent predictors, retained-row alignment and unavailable raw inputs.

For transformed LSS models (location-scale-shape models, in which each distribution parameter has its own additive predictor), `fitted_mean` is unavailable in both calibration routes. In the lognormal example, the mean is $\exp(\mu+\sigma^2/2)$, whereas the median is $\exp(\mu)$. Transforming the fitted location cannot generally recover the expectation. Untransformed LSS models retain their existing location output.

These identities establish units and preprocessing exclusion on the stated cases. They do not estimate predictive coverage or establish conditional coverage. The split-conformal marginal guarantee still requires exchangeable rows and a specification and stopping rule chosen independently of the calibration responses. The response-centering exception is recorded separately.

In [6]:
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    live = run()
for current in live["records"]:
    assert "error" not in current
    assert current["max_absolute_error"] < 1e-10
    assert current["fitted_mean_available"] is False
print("All 12 analytical and partition checks pass.")

All 12 analytical and partition checks pass.
